# deepMaze — train DRQN + DTQN (Colab Drive-backed **or** local)

Auto-detects environment via `IS_COLAB` flag set in cell 2:

- **Colab** — mounts Drive, clones the repo into `/content/deepMaze`, persists everything (MLflow, weights, replays) to `${DRIVE_BASE}`.
- **Local** — skips Drive + clone, runs from the existing repo checkout, persists everything under `<repo>/local_runs/`. Bundles also written to `<repo>/assets/` so `python web/server.py` picks them up directly.

**No GCP required either way.**

> **Heads up:** 30×60 with `partial_view=3` is hard — the agent has a 7×7 window in a ~1800-cell maze. Training takes ~1–2 h on T4 for DRQN at 6 k episodes. Drop `MAZE_WIDTH/HEIGHT` (or use the curriculum cell at the bottom) for faster smoke-tests.


## 1 — Mount Drive

In [ ]:
# Detect environment + mount Drive (Colab only).
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    IS_COLAB = False
    print("not on Colab — skipping Drive mount; will use ./local_runs/ for outputs")


## 2 — Config (Colab form)

In [ ]:
# @title Config { run: "auto" }
# ╭─ Repo + paths ──────────────────────────────────────────────────────────╮
REPO_URL    = "https://github.com/juan-garassino/deepMaze.git"  # @param {type:"string"}
REPO_BRANCH = "main"                                             # @param {type:"string"}
DRIVE_BASE  = "/content/drive/MyDrive/deepMaze"                  # @param {type:"string"}

# ╭─ Run ──────────────────────────────────────────────────────────────────╮
AGENTS_TO_RUN     = "drqn,dtqn"  # @param {type:"string"}
RUN_TAG           = "v1"          # @param {type:"string"}
MLFLOW_EXPERIMENT = "deepmaze"   # @param {type:"string"}
SEED              = 0             # @param {type:"integer"}

# NANO_LOCAL=True → when running outside Colab, auto-shrink everything below
# to nano values (8x8 maze, 200 episodes, ~2 min on CPU) — local is for
# pipeline smoke-tests, not convergence. Big training happens on Colab/RunPod.
# Set False to use the full config locally.
NANO_LOCAL = True   # @param {type:"boolean"}

# ╭─ Maze ─────────────────────────────────────────────────────────────────╮
MAZE_WIDTH   = 30    # @param {type:"integer"}
MAZE_HEIGHT  = 60    # @param {type:"integer"}
GENERATOR    = "dfs" # @param ["open", "dfs", "random"]
DENSITY      = 0.2   # @param {type:"number"}
N_TREASURES  = 10    # @param {type:"integer"}
N_LAVA       = 2     # @param {type:"integer"}
COLLECT_ALL  = False # @param {type:"boolean"}
PARTIAL      = 5     # @param {type:"integer"}
RANDOM_START = True  # @param {type:"boolean"}
BUMP_PENALTY = -0.01 # @param {type:"number"}
AUX_FEATURES   = True  # @param {type:"boolean"}
REWARD_SHAPING = True  # @param {type:"boolean"}

# ╭─ Periodic eval + curriculum gate ──────────────────────────────────────╮
EVAL_EVERY        = 0    # @param {type:"integer"}   # 0 = num_episodes//10
ADVANCE_THRESHOLD = 0.5  # @param {type:"number"}    # 0 = gate off
STAGE_MAX_REPEATS = 1    # @param {type:"integer"}

# ╭─ Generalization ───────────────────────────────────────────────────────╮
REGENERATE_EVERY = 1     # @param {type:"integer"}
EVAL_REGENERATE  = True  # @param {type:"boolean"}

# ╭─ Training budget ──────────────────────────────────────────────────────╮
NUM_EPISODES  = 3000  # @param {type:"integer"}
MAX_STEPS     = 600   # @param {type:"integer"}
EVAL_EPISODES = 50    # @param {type:"integer"}

# ╭─ Agent hyperparameter overrides (0 / 0.0 = use repo default) ──────────╮
# Epsilon decays once per EPISODE (repo default 0.995 → floor ~ep 600).
# For 3000-episode runs 0.998 stretches exploration to ~ep 1500.
# BUFFER_CAPACITY is in EPISODES for drqn/dtqn (repo default 200).
EXPLORATION_DECAY = 0.998  # @param {type:"number"}
BUFFER_CAPACITY   = 0      # @param {type:"integer"}
LEARNING_RATE     = 0.0       # @param {type:"number"}
BATCH_SIZE        = 0         # @param {type:"integer"}
SEQ_LEN           = 0         # @param {type:"integer"}
TARGET_SYNC       = 0         # @param {type:"integer"}
LSTM_HIDDEN       = 0         # @param {type:"integer"}

# ╭─ Display + showcase ───────────────────────────────────────────────────╮
PRINT_EVERY     = 25   # @param {type:"integer"}
SHOWCASE_EVERY  = 250  # @param {type:"integer"}
WINDOW          = 100  # @param {type:"integer"}
SHOWCASE_SPRITE = 12   # @param {type:"integer"}
SHOWCASE_FRAMES = 300  # @param {type:"integer"}


## 3 — Clone repo + install deps

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

if IS_COLAB:
    WORK = pathlib.Path("/content/deepMaze")
    os.chdir("/content")
    if WORK.exists():
        shutil.rmtree(WORK)
    subprocess.check_call(["git", "clone", "--depth=1", "-b", REPO_BRANCH, REPO_URL, str(WORK)])
    subprocess.run(["find", str(WORK), "-name", "__pycache__", "-type", "d",
                    "-exec", "rm", "-rf", "{}", "+"], check=False)
    os.chdir(WORK)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "-r", "requirements.txt", "mlflow==2.18.*"])
else:
    # Local: walk up from cwd until we find the repo root marker.
    WORK = pathlib.Path.cwd().resolve()
    while not (WORK / "CLAUDE.md").exists() and WORK.parent != WORK:
        WORK = WORK.parent
    if not (WORK / "CLAUDE.md").exists():
        raise RuntimeError(
            "Could not find repo root (no CLAUDE.md). "
            "Open this notebook from inside the deepMaze repo, or cd into it first."
        )
    os.chdir(WORK)
    # Assume local deps are already installed; if mlflow is missing the next
    # cell will give a clean ImportError.

for sub in ("agents", "environment", "training", "utils", "config"):
    p = str(WORK / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

# Drop cached repo modules so subsequent imports re-read from disk.
_REPO_MODS = ("train", "maze", "viz_events", "recorders", "seeding",
              "hyperparameters", "manager", "replay_buffer", "report",
              "visualizations", "drqn_agent", "dtqn_agent", "dqn_agent",
              "q_agent", "ppo_agent", "base_agent", "nets", "encoders")
for _m in list(sys.modules):
    if _m.split(".")[0] in _REPO_MODS:
        del sys.modules[_m]

if IS_COLAB:
    head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
    print(f"[colab] repo: {head}  branch: {REPO_BRANCH}")
else:
    print(f"[local] repo at {WORK}  (caches cleared)")


## 4 — Drive paths + MLflow tracking URI

In [ ]:
from pathlib import Path

# Local: persist under <repo>/local_runs/ instead of Drive.
DRIVE_BASE_P = Path(DRIVE_BASE) if IS_COLAB else (WORK / "local_runs")
MLRUNS_DIR   = DRIVE_BASE_P / "mlruns"
ASSETS_DIR   = DRIVE_BASE_P / "assets"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = f"file://{MLRUNS_DIR.as_posix()}"

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# Nano override: when local + NANO_LOCAL, shrink everything so a complete
# train→eval→bundle cycle runs in ~2 min on CPU. Goal is "did the pipeline
# work end-to-end", NOT "did the agent converge". Real training lives on
# Colab (notebook) or RunPod (runpod/Dockerfile).
if not IS_COLAB and NANO_LOCAL:
    print()
    print("⚡ NANO LOCAL OVERRIDE — pipeline smoke-test only (not convergence).")
    print("   Set NANO_LOCAL=False in cell 4 to use your full config locally.")
    MAZE_WIDTH, MAZE_HEIGHT = 8, 8
    N_TREASURES, N_LAVA     = 1, 0
    PARTIAL                 = 2
    COLLECT_ALL             = False
    NUM_EPISODES, MAX_STEPS = 200, 100
    EVAL_EPISODES           = 10
    AGENTS_TO_RUN           = "drqn"
    PRINT_EVERY             = 20
    SHOWCASE_EVERY          = 100
    EXPLORATION_DECAY       = 0.0   # repo default (per-episode 0.995)
    BUFFER_CAPACITY         = 0     # repo default (200 episodes)
    REGENERATE_EVERY        = 1
    # tiny nets so a 2015-era CPU finishes in minutes
    LSTM_HIDDEN             = 32
    BATCH_SIZE              = 4
    SEQ_LEN                 = 4
    EVAL_EPISODES           = 5

print()
print("mode    :", "colab" if IS_COLAB else "local")
print("tracking:", MLFLOW_TRACKING_URI)
print("assets  :", ASSETS_DIR)
print("budget  :", f"{NUM_EPISODES} episodes × max_steps={MAX_STEPS}  ({MAZE_WIDTH}×{MAZE_HEIGHT} maze)")


## 5 — Train one agent (function)

In [ ]:
"""Generic training driver. Reads ALL knobs from cell-4 globals — do not
edit this cell during normal iteration. Tweak cell 4 instead. Accepts an
optional warm_start_path to support curriculum learning (cell 15).
Works on Colab (Drive-backed) and locally (./local_runs/) — switches on
the IS_COLAB flag set in cell 2."""

import json
import shutil
import time
from collections import deque
from pathlib import Path

import mlflow
import torch

from maze import MazeEnvironment, RenderMaze
from recorders import ReplayRecorder
from seeding import seed_everything
from train import create_agent, evaluate_agent, simulate_episode, train_agent
from viz_events import EpisodeEvent, EventBus

try:
    from IPython.display import Image as _IPImage, display as _ipy_display
except Exception:
    _IPImage = None
    def _ipy_display(*a, **k): pass


def _fmt_eta(s: float) -> str:
    if s < 90:
        return f"{s:.0f}s"
    if s < 3600:
        return f"{s/60:.1f}m"
    return f"{s/3600:.2f}h"


def _hr(c: str = "─", w: int = 78) -> str:
    return c * w


def _agent_overrides(agent_type: str) -> dict:
    candidates = {
        "exploration_decay": EXPLORATION_DECAY,
        "learning_rate":     LEARNING_RATE,
        "batch_size":        BATCH_SIZE,
        "seq_len":           SEQ_LEN,
        "target_sync":       TARGET_SYNC,
    }
    if agent_type in ("drqn", "dtqn", "dqn"):
        candidates["buffer_capacity"] = BUFFER_CAPACITY
    if agent_type == "drqn":
        candidates["lstm_hidden"] = LSTM_HIDDEN
    return {k: v for k, v in candidates.items() if v}


def _module_of(agent):
    return getattr(agent, "model", None) or getattr(agent, "ac", None)


def _warm_start(agent, path: str) -> None:
    """Load weights into both model and target_model (if present)."""
    if path.endswith(".pkl"):  # tabular Q
        import pickle
        with open(path, "rb") as f:
            agent.Q.update(pickle.load(f))
        return
    sd = torch.load(path, map_location=getattr(agent, "device", "cpu"), weights_only=True)
    _module_of(agent).load_state_dict(sd)
    if hasattr(agent, "target_model"):
        agent.target_model.load_state_dict(sd)


def train_one(agent_type: str, run_name: str, warm_start_path: str | None = None) -> dict:
    title = f"  {agent_type.upper()}  —  {run_name}  "
    print(_hr("━"))
    print(" " * max(0, (78 - len(title)) // 2) + title)
    print(_hr("━"))

    seed_everything(SEED)

    env = MazeEnvironment(
        width=MAZE_WIDTH, height=MAZE_HEIGHT, density=DENSITY,
        generator=GENERATOR, n_lava=N_LAVA, n_treasures=N_TREASURES,
        collect_all=COLLECT_ALL, partial_view=PARTIAL, seed=SEED,
        bump_penalty=BUMP_PENALTY,
        aux_features=AUX_FEATURES, reward_shaping=REWARD_SHAPING,
    )
    overrides = _agent_overrides(agent_type)
    agent = create_agent(agent_type, env, **overrides)

    if warm_start_path:
        try:
            _warm_start(agent, warm_start_path)
            print(f"  warm start  : {warm_start_path}")
        except Exception as e:
            print(f"  warm start FAILED ({e}) — training from scratch")

    print(f"  mode        : {'colab' if IS_COLAB else 'local'}")
    print(f"  agent       : {type(agent).__name__}")
    print(f"  maze        : {MAZE_WIDTH}x{MAZE_HEIGHT}  generator={GENERATOR}  density={DENSITY}")
    print(f"  treasures   : {N_TREASURES}  lava: {N_LAVA}  collect_all: {COLLECT_ALL}  partial_view: {PARTIAL}")
    print(f"  budget      : {NUM_EPISODES} episodes  max_steps={MAX_STEPS}")
    print(f"  regen every : {REGENERATE_EVERY or 'off (single fixed maze)'}   eval_regen: {EVAL_REGENERATE}")
    print(f"  overrides   : {overrides or '(none — using repo defaults)'}")
    print(_hr())

    # Showcase dirs: in Colab use fast SSD + mirror to Drive; locally just one.
    persistent_showcase = ASSETS_DIR.parent / "showcase" / run_name
    persistent_showcase.mkdir(parents=True, exist_ok=True)
    if IS_COLAB:
        showcase_dir = Path(f"/content/showcase/{run_name}")
        showcase_dir.mkdir(parents=True, exist_ok=True)
    else:
        showcase_dir = persistent_showcase
    sprites = RenderMaze.placeholder_sprites(SHOWCASE_SPRITE)

    def render_snapshot(ep: int) -> Path:
        agent.set_deterministic(True)
        try:
            _, positions, _, _ = simulate_episode(env, agent, max_steps=MAX_STEPS, at_start=True)
        finally:
            agent.set_deterministic(False)
        full_frames = [env.maze.copy() for _ in positions]
        rm = RenderMaze(sprites)
        ReplayRecorder(rm).feed(full_frames, positions, None)
        out = showcase_dir / f"ep{ep:05d}.webp"
        rm.save(str(out), fmt="webp", sprite_size=SHOWCASE_SPRITE, max_frames=SHOWCASE_FRAMES)
        if showcase_dir != persistent_showcase:
            try:
                shutil.copy(out, persistent_showcase / out.name)
            except Exception as e:
                print(f"  [showcase] persistent copy failed: {e}")
        return out

    bus = EventBus()

    def _on_ep_mlflow(ev: EpisodeEvent):
        mlflow.log_metrics(
            {"episode_reward": ev.total_reward,
             "episode_length": ev.length,
             "epsilon": ev.epsilon},
            step=ev.episode,
        )

    reward_buf  = deque(maxlen=WINDOW)
    length_buf  = deque(maxlen=WINDOW)
    success_buf = deque(maxlen=WINDOW)
    t0 = time.time()

    def _on_ep_print(ev: EpisodeEvent):
        reward_buf.append(ev.total_reward)
        length_buf.append(ev.length)
        success_buf.append(1 if ev.success else 0)

        elapsed   = time.time() - t0
        eps_per_s = (ev.episode + 1) / max(elapsed, 1e-6)
        eta       = (NUM_EPISODES - ev.episode - 1) / max(eps_per_s, 1e-6)

        if ev.episode % PRINT_EVERY == 0 or ev.episode == NUM_EPISODES:
            avg_r = sum(reward_buf) / len(reward_buf)
            succ  = 100.0 * sum(success_buf) / len(success_buf)
            extra = f" loss={ev.loss:.4f}" if ev.loss is not None else ""
            print(
                f"  ep {ev.episode:>5}/{NUM_EPISODES}  "
                f"R={ev.total_reward:+6.2f}  R̄{WINDOW}={avg_r:+6.2f}  succ%={succ:5.1f}  "
                f"len={ev.length:>4}  ε={ev.epsilon:.3f}{extra}  "
                f"[{eps_per_s:5.1f} ep/s  ETA {_fmt_eta(eta)}]",
                flush=True,
            )

        if ev.episode > 0 and (ev.episode % SHOWCASE_EVERY == 0):
            avg_r = sum(reward_buf) / len(reward_buf)
            avg_l = sum(length_buf) / len(length_buf)
            succ  = 100.0 * sum(success_buf) / len(success_buf)
            print()
            print(_hr("╌"))
            print(f"  ▣  SHOWCASE  ep {ev.episode}/{NUM_EPISODES}  ({100*ev.episode/NUM_EPISODES:.1f}%)")
            print(f"     last {WINDOW}: R̄={avg_r:+.2f}   len̄={avg_l:.0f}   success={succ:.1f}%")
            print(f"     elapsed {_fmt_eta(elapsed)}   ETA {_fmt_eta(eta)}   throughput {eps_per_s:.1f} ep/s")
            snap = render_snapshot(ev.episode)
            print(f"     replay → {snap.name}  ({snap.stat().st_size//1024} KB)")
            if _IPImage is not None:
                _ipy_display(_IPImage(filename=str(snap)))
            print(_hr("╌"))
            print()

    bus.subscribe(_on_ep_mlflow)
    bus.subscribe(_on_ep_print)

    best = {"succ": -1.0, "sd": None, "episode": None}

    def _on_eval(ep, m):
        mlflow.log_metrics({f"periodic_{k}": v for k, v in m.items()}, step=ep)
        print(f"  ◈ eval @ {ep}: succ={m['success_rate']:.1%} "
              f"R̄={m['mean_reward']:+.2f}", flush=True)
        if m["success_rate"] > best["succ"]:
            best["succ"] = m["success_rate"]
            best["episode"] = ep
            module = _module_of(agent)
            if module is not None:
                best["sd"] = {k: v.detach().cpu().clone()
                              for k, v in module.state_dict().items()}

    eval_every = EVAL_EVERY or max(50, NUM_EPISODES // 10)

    params = dict(
        agent_type=agent_type, run_name=run_name,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        generator=GENERATOR, density=DENSITY,
        n_treasures=N_TREASURES, n_lava=N_LAVA, collect_all=COLLECT_ALL,
        partial=PARTIAL, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, seed=SEED,
        random_start=RANDOM_START, bump_penalty=BUMP_PENALTY,
        aux_features=AUX_FEATURES, reward_shaping=REWARD_SHAPING,
        regenerate_every=REGENERATE_EVERY, eval_regenerate=EVAL_REGENERATE,
        warm_start_from=warm_start_path or "",
        mode="colab" if IS_COLAB else "local",
        **overrides,
    )

    train_t0 = time.time()
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        train_agent(env, agent, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS, bus=bus,
                    random_start=RANDOM_START,
                    regenerate_every=(REGENERATE_EVERY or None),
                    eval_every=eval_every, eval_episodes=10, on_eval=_on_eval)
        mean_reward, mean_length, success_rate = evaluate_agent(
            env, agent, num_episodes=EVAL_EPISODES, max_steps=MAX_STEPS,
            regenerate_each=EVAL_REGENERATE,
        )
        mlflow.log_metrics({
            "eval_mean_reward": mean_reward,
            "eval_mean_length": mean_length,
            "eval_success_rate": success_rate,
        })
        run_id = run.info.run_id

    print()
    print(_hr("━"))
    print(f"  ✓ {agent_type.upper()} done in {_fmt_eta(time.time() - train_t0)}")
    print(f"    eval: reward {mean_reward:+.2f}   length {mean_length:.0f}   success {success_rate:.1%}")
    print(_hr("━"))

    local_out      = WORK / "assets" / run_name if not IS_COLAB else Path(f"assets/{run_name}")
    persistent_out = ASSETS_DIR / run_name
    for out in (local_out, persistent_out):
        (out / "viz").mkdir(parents=True, exist_ok=True)

    config = dict(
        agent_type=agent_type, net=None,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        n_agents=1, density=DENSITY, generator=GENERATOR,
        no_ensure_solvable=False, n_lava=N_LAVA, lava_reward=-1.0,
        bump_penalty=BUMP_PENALTY,
        aux_features=AUX_FEATURES, reward_shaping=REWARD_SHAPING,
        partial=PARTIAL, n_treasures=N_TREASURES, collect_all=COLLECT_ALL,
        seed=SEED, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, learning_rate=None, discount_factor=0.99,
        image_path=None, sprite_files=["sprites.png"], sprite_size=32,
        replay_fmt="webp", frame_skip=1, max_frames=None,
        policy_snapshot_every=50, live=False, live_web=False, web_port=8000,
        run_id=run_id, run_name=run_name,
        random_start=RANDOM_START, resume=None, eval_maze="same", eval_seeds=1,
        agent_hp=overrides,
    )
    (local_out / "config.json").write_text(json.dumps(config, indent=2))

    module = _module_of(agent)
    if module is not None:
        torch.save(module.state_dict(), local_out / "model.pt")
    else:  # tabular Q
        import pickle
        (local_out / "model.pkl").write_bytes(pickle.dumps(dict(agent.Q)))
    best_model_path = None
    if best["sd"] is not None:
        torch.save(best["sd"], local_out / "model.best.pt")
        best_model_path = str(local_out / "model.best.pt")
        print(f"  best snapshot: succ={best['succ']:.1%} @ ep {best['episode']}")

    final_snap = render_snapshot(NUM_EPISODES)
    shutil.copy(final_snap, local_out / "viz" / "replay.webp")
    print(f"  final replay → {local_out / 'viz' / 'replay.webp'}")
    if _IPImage is not None:
        _ipy_display(_IPImage(filename=str(local_out / "viz" / "replay.webp")))

    if local_out != persistent_out:
        if persistent_out.exists():
            shutil.rmtree(persistent_out)
        shutil.copytree(local_out, persistent_out)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifacts(str(local_out), artifact_path=f"assets/{run_name}")

    print(f"  bundle → {persistent_out}")
    return {
        "agent_type": agent_type,
        "run_name": run_name,
        "run_id": run_id,
        "eval_success_rate": success_rate,
        "eval_mean_reward": mean_reward,
        "best_success_rate": best["succ"],
        "drive_path": str(persistent_out),
        "showcase_dir": str(persistent_showcase),
        "model_path": str(persistent_out / ("model.pt" if module is not None
                                            else "model.pkl")),
        "best_model_path": (str(persistent_out / "model.best.pt")
                            if best_model_path else None),
    }


## 6 — Train DRQN then DTQN sequentially

In [ ]:
agents = [a.strip() for a in AGENTS_TO_RUN.split(",") if a.strip()]
results = []
for agent_type in agents:
    run_name = f"{agent_type}_{RUN_TAG}"
    results.append(train_one(agent_type, run_name))

print("\n=== summary ===")
for r in results:
    print(f"  {r['agent_type']:5s}  {r['run_name']:20s}  success={r['eval_success_rate']:.2%}  → {r['drive_path']}")


## 7 — Use the bundles locally

On your laptop:

```bash
# Copy a bundle from Drive into the repo's assets/
rsync -a "<google-drive-folder>/deepMaze/assets/drqn_v1/" assets/drqn_v1/

# Then either:
python web/server.py --port 8000    # → http://localhost:8000  → pretrained dropdown
# or:
python main.py --agent_type drqn --resume assets/drqn_v1/model.pt --num_episodes 0
```

To browse MLflow runs locally after copying `mlruns/` down from Drive:

```bash
mlflow ui --backend-store-uri "file://$(pwd)/mlruns"
```


## 8 — Curriculum (optional, alternative to cell 6)

When training on a big maze from scratch fails because the random walker can't stumble on a treasure, **curriculum learning** fixes it: start on a small maze (random exploration finds the goal often), then transfer those weights to a larger maze.

Works here because `PARTIAL` keeps the observation shape constant — the network sees an 11×11 window whether the underlying maze is 10×20 or 30×60 — so weights load cleanly between stages.

Run `results = run_curriculum("drqn")` (or `"dtqn"`) in the next cell. Edit `CURRICULUM_STAGES` to change the schedule.


In [ ]:
# @title Curriculum schedule { run: "auto" }
# Each stage: (width, height, n_treasures, num_episodes, max_steps).
# Tune freely — sizes grow, treasure density loosens, max_steps follows
# the maze diameter. Reasonable for DRQN on a T4 GPU.
CURRICULUM_STAGES = [
    (10, 20, 5,  800, 200),
    (20, 40, 8, 1200, 400),
    (30, 60, 10, 2000, 600),
]


def run_curriculum(agent_type: str, run_tag: str = "curr") -> list[dict]:
    """Train through CURRICULUM_STAGES with weight transfer between stages."""
    global MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS
    saved = (MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS)

    results = []
    warm = None
    try:
        for i, (w, h, nt, neps, mx) in enumerate(CURRICULUM_STAGES):
            MAZE_WIDTH, MAZE_HEIGHT = w, h
            N_TREASURES = nt
            NUM_EPISODES, MAX_STEPS = neps, mx

            res = None
            for attempt in range(1 + max(0, STAGE_MAX_REPEATS - 1)):
                run_name = f"{agent_type}_{run_tag}_s{i}_{w}x{h}" + (
                    f"_r{attempt}" if attempt else "")
                print()
                print("█" * 78)
                print(f"  CURRICULUM stage {i+1}/{len(CURRICULUM_STAGES)}  —  "
                      f"{w}x{h}  ({nt} treasures, {neps} eps, max_steps={mx})")
                if warm:
                    print(f"  warm-starting from: {warm}")
                print("█" * 78)

                res = train_one(agent_type, run_name, warm_start_path=warm)
                results.append(res)
                gate = max(res["eval_success_rate"], res["best_success_rate"])
                if not ADVANCE_THRESHOLD or gate >= ADVANCE_THRESHOLD:
                    break
                warm = res["best_model_path"] or res["model_path"]
                print(f"  ⟳ stage below gate ({gate:.1%} < "
                      f"{ADVANCE_THRESHOLD:.1%}) — retrying from best")
            gate = max(res["eval_success_rate"], res["best_success_rate"])
            if ADVANCE_THRESHOLD and gate < ADVANCE_THRESHOLD:
                print(f"  ✗ stage {i} never passed the gate — stopping curriculum")
                break
            warm = res["best_model_path"] or res["model_path"]
    finally:
        MAZE_WIDTH, MAZE_HEIGHT, N_TREASURES, NUM_EPISODES, MAX_STEPS = saved

    print()
    print("=" * 78)
    print("  CURRICULUM SUMMARY")
    print("=" * 78)
    for r in results:
        print(f"  {r['run_name']:40s}  success={r['eval_success_rate']:.2%}")
    return results


# Uncomment to run:
# results = run_curriculum("drqn")
# results = run_curriculum("dtqn", run_tag="curr_dtqn")
